In [1]:
# ============================================
# FRAME EXTRACTION FROM VIDEOS (GOOGLE COLAB)
# Reads videos from Google Drive and saves
# extracted frames back to Drive by class.
# ============================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import cv2
from pathlib import Path

In [3]:
# -----------------------------
# SET YOUR BASE PROJECT FOLDER
# -----------------------------
BASE_DIR = "/content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization"

RAW_VIDEOS_DIR = os.path.join(BASE_DIR, "raw_videos_test")
EXTRACTED_FRAMES_DIR = os.path.join(BASE_DIR, "extracted_frames_test")

os.makedirs(EXTRACTED_FRAMES_DIR, exist_ok=True)

print("RAW_VIDEOS_DIR:", RAW_VIDEOS_DIR)
print("EXTRACTED_FRAMES_DIR:", EXTRACTED_FRAMES_DIR)

RAW_VIDEOS_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/raw_videos_test
EXTRACTED_FRAMES_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/extracted_frames_test


In [4]:
# -----------------------------
# CLASS FOLDERS TO PROCESS
# -----------------------------
class_names = [
    "floor1_hallway",
    "floor2_hallway",
    "floor3_hallway",
    "front_lobby",
    "laundry",
    "ccu_lounge",
    "floor1_elevator_landmark",
    "floor2_elevator_landmark",
    "floor3_elevator_landmark"
]

In [5]:
# -----------------------------
# EXTRACTION SETTINGS
# -----------------------------
# Save 1 frame every `seconds_between_frames`
seconds_between_frames = 0.25

# Supported video extensions
video_extensions = {".mp4", ".mov", ".avi", ".mkv"}

In [6]:
def extract_frames_from_video(video_path, output_folder, seconds_between_frames=1.0):
    """
    Extract frames from a single video at a fixed time interval.
    Saves frames into output_folder.

    Returns:
        saved_count (int): Number of frames saved
    """
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[WARNING] Could not open video: {video_path}")
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        print(f"[WARNING] Invalid FPS for video: {video_path}")
        cap.release()
        return 0

    frame_interval = max(1, int(round(fps * seconds_between_frames)))

    video_name = Path(video_path).stem
    frame_idx = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            frame_filename = f"{video_name}_frame_{saved_count:04d}.jpg"
            frame_path = os.path.join(output_folder, frame_filename)
            cv2.imwrite(frame_path, frame)
            saved_count += 1

        frame_idx += 1

    cap.release()
    return saved_count

In [7]:
# -----------------------------
# BATCH EXTRACTION FOR ALL CLASSES
# -----------------------------
summary = {}

for class_name in class_names:
    class_video_dir = os.path.join(RAW_VIDEOS_DIR, class_name)
    class_output_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)

    os.makedirs(class_output_dir, exist_ok=True)

    if not os.path.exists(class_video_dir):
        print(f"[WARNING] Missing class folder: {class_video_dir}")
        summary[class_name] = 0
        continue

    video_files = [
        os.path.join(class_video_dir, f)
        for f in os.listdir(class_video_dir)
        if Path(f).suffix.lower() in video_extensions
    ]

    total_saved = 0

    print(f"\nProcessing class: {class_name}")
    print(f"Found {len(video_files)} video(s)")

    for video_path in sorted(video_files):
        saved = extract_frames_from_video(
            video_path=video_path,
            output_folder=class_output_dir,
            seconds_between_frames=seconds_between_frames
        )
        print(f"  Saved {saved} frame(s) from {os.path.basename(video_path)}")
        total_saved += saved

    summary[class_name] = total_saved

print("\n=== EXTRACTION SUMMARY ===")
for class_name, count in summary.items():
    print(f"{class_name}: {count} frames")


Processing class: floor1_hallway
Found 5 video(s)
  Saved 19 frame(s) from floor1_hallway_test_1.MOV
  Saved 5 frame(s) from floor1_hallway_test_2.MOV
  Saved 6 frame(s) from floor1_hallway_test_3.MOV
  Saved 11 frame(s) from floor1_hallway_test_4.MOV
  Saved 10 frame(s) from floor1_hallway_test_5.MOV

Processing class: floor2_hallway
Found 9 video(s)
  Saved 44 frame(s) from floor2_hallway_test_1.MOV
  Saved 33 frame(s) from floor2_hallway_test_2.MOV
  Saved 7 frame(s) from floor2_hallway_test_3.MOV
  Saved 45 frame(s) from floor2_hallway_test_4.MOV
  Saved 74 frame(s) from floor2_hallway_test_5.MOV
  Saved 26 frame(s) from floor2_hallway_test_6.MOV
  Saved 33 frame(s) from floor2_hallway_test_7.MOV
  Saved 29 frame(s) from floor2_hallway_test_8.MOV
  Saved 13 frame(s) from floor2_hallway_test_9.MOV

Processing class: floor3_hallway
Found 4 video(s)
  Saved 20 frame(s) from floor3_hallway_test_1.MOV
  Saved 41 frame(s) from floor3_hallway_test_2.MOV
  Saved 32 frame(s) from floor3_ha

In [8]:
# -----------------------------
# CHECK TOTAL FRAMES SAVED
# -----------------------------
total_frames = 0

for class_name in class_names:
    class_output_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)
    if os.path.exists(class_output_dir):
        num_images = len([
            f for f in os.listdir(class_output_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        print(f"{class_name}: {num_images} extracted images")
        total_frames += num_images

print("\nTotal extracted images:", total_frames)

floor1_hallway: 51 extracted images
floor2_hallway: 304 extracted images
floor3_hallway: 144 extracted images
front_lobby: 70 extracted images
laundry: 65 extracted images
ccu_lounge: 70 extracted images
floor1_elevator_landmark: 47 extracted images
floor2_elevator_landmark: 72 extracted images
floor3_elevator_landmark: 52 extracted images

Total extracted images: 875
